# COMP5606 Fall 2026 — Local Chatbot Project
## Milestone M1: Scope, data and design

**Group:** `G8`  **Domain:** `Football rules assistant (IFAB Laws of the Game 2026/27)`  **Submission date:** `10/10/2026`

| Member | Student ID | Component owned |
|---|---|---|
| Abdullah Al-Sharafi | 135631 | data pipeline + evaluation harness |
| Yousef Roudani | 156565 | retrieval |
| Omar Ahmed AL-Hamimi | 137581 | response generation |
> The notebook must run top-to-bottom with **the network switched off** (Rule 1). Submit it together with your 2–3 page proposal.

## 1. Task definition


| Item | Our answer |
|---|---|
| **Users and need** | Amateur and grassroots players, coaches, youth referees and fans who want to know what the official rules say about a match situation (offside, handball, restarts, cards, timing) without reading the full rulebook. |
| **Coverage** | IFAB *Laws of the Game 2026/27* (English edition, effective 1 July 2026): Laws 1–17 and the "Law changes 2026/27" section. This is the only source loaded for M1. Optional second source, to be added in M2 only if needed: IFAB *Football Rules – simplified Laws of the Game* (footballrules.com, 2026/27 season). |
| **Question types** | **T1** fact lookup, answered with Law and page. **T2** several facts or a condition, combining passages (e.g. offence + restart + card). **T3** paraphrases and informal football wording ("pen", "handball", "the wall"). **T4** ambiguous: asks one clarifying question, then answers. **T5** out of scope: declines and says why. **T6** follow-up turns: resolves "and in extra time?" using the previous turn. |
| **Exclusions** | Match results, news, player or team statistics; judging real incidents from specific matches or videos; betting; competition-specific rules not in the Laws (e.g. one league's own regulations); futsal and beach soccer; general chat. |
| **Behaviour rules** | **Answer** only when a retrieved passage supports it, citing source, Law and page (e.g. "S1, Law 11 – Offside, p. 105"). **Clarify** with one question when the answer depends on a missing detail (which restart, where the player was). **Decline** when the question is out of scope or retrieval confidence is below a threshold, saying why and what it can answer. It never invents rules and **is not an official decision maker**: it explains the Laws, and the referee's on-field decision is final. |
| **Language** | English only (official IFAB English edition), including informal football terms. |
| **Hardware** | Windows 11 laptop, AMD64 CPU (16 logical cores), 16.5 GB RAM, no CUDA GPU (see Section 2). |

**10 representative user questions:**
1. How long is each half of a football match? *(T1)*
2. How far away must opponents be at a free kick? *(T1)*
3. Does the whole ball have to cross the line for a goal? *(T1)*
4. I received the ball straight from a corner while behind the last defender. Is that offside, and would it be different from a free kick? *(T2)*
5. A defender stops a certain goal by handling the ball on the line. What happens? *(T2)*
6. If just some of the ball is past the goal line, does the goal count? *(T3, paraphrase of 3)*
7. Is it offside? *(T4)*
8. Where must the other players stand at a kick? *(T4)*
9. Who won the last World Cup? *(T5)*
10. After question 1: "And if the match goes to extra time?" *(T6)*

## 2. Environment, hardware and offline check

In [38]:
import sys, platform, os, json, time, socket, hashlib, re
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd

PROJECT = Path(".").resolve()
DATA = PROJECT / "data"
CORPUS_DIR = DATA / "corpus"      # put your source documents here (.txt / .md; .pdf if pypdf is installed)
EVAL_DIR = DATA / "eval"
OUT = PROJECT / "outputs"
for p in (CORPUS_DIR, EVAL_DIR, OUT):
    p.mkdir(parents=True, exist_ok=True)

SEED = 5606
np.random.seed(SEED)
print("Python", sys.version.split()[0], "|", platform.platform())

Python 3.13.7 | Windows-11-10.0.26200-SP0


In [39]:
def hardware_report():
    rep = {"machine": platform.machine(), "processor": platform.processor() or "n/a",
           "cpu_count": os.cpu_count()}
    try:
        import psutil
        rep["ram_gb"] = round(psutil.virtual_memory().total / 1e9, 1)
    except ImportError:
        rep["ram_gb"] = "install psutil to report"
    try:
        import torch
        rep["cuda_gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
        rep["apple_mps"] = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
    except ImportError:
        rep["gpu"] = "torch not installed (fine for CPU-only designs)"
    return rep

HARDWARE = hardware_report()
HARDWARE

{'machine': 'AMD64',
 'processor': 'AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD',
 'cpu_count': 16,
 'ram_gb': 16.5,
 'cuda_gpu': 'none',
 'apple_mps': False}

In [40]:
def network_is_off(host="8.8.8.8", port=53, timeout=2.0):
    # Returns True if an outside connection FAILS (what we want during the demo).
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return False
    except OSError:
        return True

print("Network switched off:", network_is_off())
print("For the M1 feasibility demo, switch your network off and re-run this notebook from the top.")

Network switched off: True
For the M1 feasibility demo, switch your network off and re-run this notebook from the top.


## 3. Source inventory
One row per source document. Only use public or properly authorised sources, and remove personal data.

In [41]:
# Download the PDFs once while online and save them in data/corpus/ as:
#   S1_laws_of_the_game_2026_27.pdf    (theifab.com -> Downloads -> Laws of the Game 2026/27, English)
#   S2_football_rules_simplified.pdf   (optional: footballrules.com -> PDF downloads)
inventory = pd.DataFrame([
    {"source_id": "S1", "title": "Laws of the Game 2026/27", "owner": "The IFAB",
     "url_or_origin": "https://www.theifab.com (Downloads)", "version_or_date": "2026/27, effective 2026-07-01",
     "licence_or_permission": "Free download; reproduction needs IFAB permission, so used locally for coursework only, not redistributed",
     "format": "pdf", "language": "en", "pages": "260", "known_gaps": "Competition-specific rules not included; diagrams lost in text extraction"},
    {"source_id": "S2", "title": "Football Rules - simplified Laws of the Game", "owner": "The IFAB",
     "url_or_origin": "https://footballrules.com (PDF downloads)", "version_or_date": "2026/27 season",
     "licence_or_permission": "Free download, local coursework use only (check terms of service)",
     "format": "pdf", "language": "en", "pages": "101", "known_gaps": "Simplified, so less detail than S1 (optional source, not loaded in M1)"},
])
inventory.to_csv(OUT / "source_inventory.csv", index=False)
inventory

,source_id,title,owner,url_or_origin,version_or_date,licence_or_permission,format,language,pages,known_gaps
0,S1,Laws of the Game 2026/27,The IFAB,https://www.theifab.com (Downloads),"2026/27, effective 2026-07-01",Free download; reproduction needs IFAB permiss...,pdf,en,260,Competition-specific rules not included; diagr...
1,S2,Football Rules - simplified Laws of the Game,The IFAB,https://footballrules.com (PDF downloads),2026/27 season,"Free download, local coursework use only (chec...",pdf,en,101,"Simplified, so less detail than S1 (optional s..."


## 4. Corpus loading and chunking
Each PDF is read page by page, so every chunk keeps its `source_id` and page number (`source_location`).
Pages are then split into sentence-aware chunks. In M2 we plan to chunk by Law and sub-heading instead.

In [42]:


from pypdf import PdfReader   # pip install pypdf

def read_pages(path: Path):
    # Returns a list of (page_number, text). Text files count as a single page.
    if path.suffix.lower() == ".pdf":
        return [(i + 1, page.extract_text() or "") for i, page in enumerate(PdfReader(path).pages)]
    return [(1, path.read_text(encoding="utf-8", errors="ignore"))]

def clean(text: str) -> str:
    text = re.sub(r"Laws of the Game 20\d\d/\d\d\s*\|?", " ", text)   # drop running page headers
    text = re.sub(r"-\n(\w)", r"\1", text)                            # re-join hyphenated words
    text = re.sub(r"[ \t]+", " ", text)
    return re.sub(r"\n{2,}", "\n", text).strip()

def chunk_text(text: str, source_id: str, page: int, max_words=150, overlap=30):
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    chunks, buf, n = [], [], 0
    def emit():
        return {"chunk_id": f"{source_id}-p{page:03d}-{n:02d}", "source_id": source_id,
                "source_location": f"p. {page}", "text": " ".join(buf)}
    for s in sents:
        if buf and len(" ".join(buf + [s]).split()) > max_words:
            chunks.append(emit()); n += 1
            keep, count = [], 0
            for prev in reversed(buf):
                count += len(prev.split())
                if count > overlap: break
                keep.insert(0, prev)
            buf = keep
        buf.append(s)
    if buf:
        chunks.append(emit())
    return chunks

# --- Check the corpus folder ---
files = [f for f in sorted(CORPUS_DIR.iterdir()) if f.suffix.lower() in {".txt", ".md", ".pdf"}]
print("Corpus folder:", CORPUS_DIR)
print("Files found:", [f.name for f in files] or "NONE")

# --- Load and chunk ---
rows = []
for path in files:
    source_id = path.stem.split("_")[0]          # file name must start with its source_id, e.g. S1_laws.pdf
    pages = read_pages(path)
    kept = 0
    for page, text in pages:
        text = clean(text)
        if len(text.split()) > 15:               # skip near-empty pages (covers, photos)
            rows += chunk_text(text, source_id, page)
            kept += 1
    print(f"  {path.name}: {len(pages)} pages, {kept} pages with text")

chunks = pd.DataFrame(rows, columns=["chunk_id", "source_id", "source_location", "text"])
if chunks.empty:
    raise RuntimeError(f"No chunks created. Put your PDFs in {CORPUS_DIR} "
                       "(named like S1_laws_of_the_game_2026_27.pdf) and check that pypdf can read their text.")

chunks.to_csv(OUT / "chunks_sample.csv", index=False)
print(f"\n{chunks.source_id.nunique()} documents -> {len(chunks)} chunks; "
      f"median {chunks.text.str.split().str.len().median():.0f} words per chunk")
chunks.head()

Corpus folder: C:\Users\Omarh\Desktop\NLP\NLP_project\data\corpus
Files found: ['S1_laws_of_the_game_2026_27.pdf']
  S1_laws_of_the_game_2026_27.pdf: 260 pages, 186 pages with text

1 documents -> 375 chunks; median 119 words per chunk


,chunk_id,source_id,source_location,text
0,S1-p004-00,S1,p. 4,4\nEffective from 1 st July 2026\nThis booklet...
1,S1-p006-00,S1,p. 6,6\nContents\nAbout the Laws\nNotes and modific...
2,S1-p007-00,S1,p. 7,7\n Contents\n15 The Throw-in\n16 The Goal Kic...
3,S1-p011-00,S1,p. 11,11\n Football is the greatest sport on earth. ...
4,S1-p011-01,S1,p. 11,The best \nmatches are those where the referee...


## 5. Evaluation question sample (at least 10, hand-checked)
Required columns plus three extra ones: `split` (dev/test; paraphrase pairs and dialogues stay in the same split),
`paraphrase_of` (T3: the qid of the original question) and `context` (T6: the previous turn).
These 13 items belong to the **development** split. The frozen test set (≥ 50) is written separately in M2.

In [43]:

eval_sample = pd.DataFrame([
 {'qid': 'Q001', 'test_case': 'T1', 'question': 'How long is each half of a match?', 'reference_answer': 'Two equal halves of 45 minutes.', 'source_id': 'S1', 'source_location': 'Law 7, p. 93', 'expected_behaviour': 'answer', 'paraphrase_of': '', 'context': '', 'split': 'dev', 'written_by': 'Abdullah Al-Sharafi', 'checked_by': 'Yousef Roudani', 'ai_drafted': True},
 {'qid': 'Q002', 'test_case': 'T1', 'question': 'How far must opponents be from the ball at a free kick?', 'reference_answer': 'At least 9.15 m (10 yds) from the ball until it is in play, unless they are on their own goal line between the goalposts.', 'source_id': 'S1', 'source_location': 'Law 13, p. 132', 'expected_behaviour': 'answer', 'paraphrase_of': '', 'context': '', 'split': 'dev', 'written_by': 'Abdullah Al-Sharafi', 'checked_by': 'Yousef Roudani', 'ai_drafted': True},
 {'qid': 'Q003', 'test_case': 'T1', 'question': 'Does the whole ball have to cross the goal line for a goal?', 'reference_answer': 'Yes. A goal is scored when the whole ball passes over the goal line between the posts and under the crossbar.', 'source_id': 'S1', 'source_location': 'Law 10, p. 103', 'expected_behaviour': 'answer', 'paraphrase_of': '', 'context': '', 'split': 'dev', 'written_by': 'Abdullah Al-Sharafi', 'checked_by': 'Yousef Roudani', 'ai_drafted': True},
 {'qid': 'Q004', 'test_case': 'T1', 'question': 'What size must the ball be?', 'reference_answer': 'Circumference between 68 cm and 70 cm.', 'source_id': 'S1', 'source_location': 'Law 2, p. 55', 'expected_behaviour': 'answer', 'paraphrase_of': '', 'context': '', 'split': 'dev', 'written_by': 'Abdullah Al-Sharafi', 'checked_by': 'Yousef Roudani', 'ai_drafted': True},
 {'qid': 'Q005', 'test_case': 'T2', 'question': 'I received the ball straight from a corner while behind the last defender. Is that offside, and would it be different from a free kick?', 'reference_answer': 'No offside offence directly from a corner kick (also goal kick and throw-in). From a free kick, offside can be penalised.', 'source_id': 'S1', 'source_location': 'Law 11, pp. 109-111', 'expected_behaviour': 'answer', 'paraphrase_of': '', 'context': '', 'split': 'dev', 'written_by': 'Abdullah Al-Sharafi', 'checked_by': 'Yousef Roudani', 'ai_drafted': True},
 {'qid': 'Q006', 'test_case': 'T2', 'question': 'A defender stops a certain goal by handling the ball on the line. What happens?', 'reference_answer': "If the handball is deliberate, the defender is sent off for denying a goal by a deliberate handball offence, and a penalty kick is awarded if it happened in the defender's own penalty area (otherwise a direct free kick). If the handball is non-deliberate and a penalty kick is awarded, the defender is only cautioned (yellow card).", 'source_id': 'S1', 'source_location': 'Law 12, pp. 122-124; Law 14, p. 135', 'expected_behaviour': 'answer', 'paraphrase_of': '', 'context': '', 'split': 'dev', 'written_by': 'Yousef Roudani', 'checked_by': 'Omar Ahmed AL-Hamimi', 'ai_drafted': True},
 {'qid': 'Q007', 'test_case': 'T3', 'question': 'How many minutes does one half of a game last?', 'reference_answer': 'Two equal halves of 45 minutes.', 'source_id': 'S1', 'source_location': 'Law 7, p. 93', 'expected_behaviour': 'answer', 'paraphrase_of': 'Q001', 'context': '', 'split': 'dev', 'written_by': 'Yousef Roudani', 'checked_by': 'Omar Ahmed AL-Hamimi', 'ai_drafted': True},
 {'qid': 'Q008', 'test_case': 'T3', 'question': 'If just some of the ball is past the goal line, does the goal count?', 'reference_answer': 'No. The whole ball must pass over the goal line between the posts and under the crossbar.', 'source_id': 'S1', 'source_location': 'Law 10, p. 103', 'expected_behaviour': 'answer', 'paraphrase_of': 'Q003', 'context': '', 'split': 'dev', 'written_by': 'Yousef Roudani', 'checked_by': 'Omar Ahmed AL-Hamimi', 'ai_drafted': True},
 {'qid': 'Q009', 'test_case': 'T4', 'question': 'Is it offside?', 'reference_answer': 'Clarify: where was the player when the ball was played by a teammate, and how was the ball received (open play, goal kick, corner kick or throw-in)?', 'source_id': 'S1', 'source_location': 'Law 11, pp. 109-111', 'expected_behaviour': 'clarify', 'paraphrase_of': '', 'context': '', 'split': 'dev', 'written_by': 'Yousef Roudani', 'checked_by': 'Omar Ahmed AL-Hamimi', 'ai_drafted': True},
 {'qid': 'Q010', 'test_case': 'T4', 'question': 'Where must the other players stand at a kick?', 'reference_answer': 'Clarify: which kick (kick-off, free kick, penalty kick, corner kick or goal kick)?', 'source_id': 'S1', 'source_location': 'Laws 8, 13, 14, 16, 17', 'expected_behaviour': 'clarify', 'paraphrase_of': '', 'context': '', 'split': 'dev', 'written_by': 'Omar Ahmed AL-Hamimi', 'checked_by': 'Abdullah Al-Sharafi', 'ai_drafted': True},
 {'qid': 'Q011', 'test_case': 'T5', 'question': 'Who won the last World Cup?', 'reference_answer': 'Decline: match results are not covered by the Laws of the Game.', 'source_id': '', 'source_location': '', 'expected_behaviour': 'decline', 'paraphrase_of': '', 'context': '', 'split': 'dev', 'written_by': 'Omar Ahmed AL-Hamimi', 'checked_by': 'Abdullah Al-Sharafi', 'ai_drafted': True},
 {'qid': 'Q012', 'test_case': 'T5', 'question': "Was the penalty in yesterday's match the right decision?", 'reference_answer': 'Decline: the chatbot explains the Laws but does not judge real match incidents.', 'source_id': '', 'source_location': '', 'expected_behaviour': 'decline', 'paraphrase_of': '', 'context': '', 'split': 'dev', 'written_by': 'Omar Ahmed AL-Hamimi', 'checked_by': 'Abdullah Al-Sharafi', 'ai_drafted': True},
 {'qid': 'Q013', 'test_case': 'T6', 'question': 'And if the match goes to extra time?', 'reference_answer': 'Extra time is two equal periods, each not more than 15 minutes.', 'source_id': 'S1', 'source_location': 'Law 10, p. 103', 'expected_behaviour': 'answer', 'paraphrase_of': '', 'context': 'Q001: How long is each half of a match?', 'split': 'dev', 'written_by': 'Omar Ahmed AL-Hamimi', 'checked_by': 'Abdullah Al-Sharafi', 'ai_drafted': True},
])
REQUIRED = ["qid","test_case","question","reference_answer","source_id","source_location",
            "expected_behaviour","written_by","checked_by","ai_drafted"]

def validate_eval(df):
    problems = [f"missing column: {c}" for c in REQUIRED if c not in df.columns]
    if not problems:
        problems += [f"{q}: bad test_case" for q, t in zip(df.qid, df.test_case) if t not in {"T1","T2","T3","T4","T5","T6"}]
        problems += [f"{q}: bad expected_behaviour" for q, b in zip(df.qid, df.expected_behaviour) if b not in {"answer","clarify","decline"}]
        problems += [f"duplicate qid: {q}" for q in df.qid[df.qid.duplicated()]]
        problems += [f"{q}: answerable item has no source_location" for q, b, s in
                     zip(df.qid, df.expected_behaviour, df.source_location) if b == "answer" and not s]
    return problems

problems = validate_eval(eval_sample)
problems += [f"{q}: paraphrase_of {p} not found" for q, p in zip(eval_sample.qid, eval_sample.paraphrase_of)
             if p and p not in set(eval_sample.qid)]
problems += [f"{q}: T6 item has no context" for q, t, c in zip(eval_sample.qid, eval_sample.test_case, eval_sample.context)
             if t == "T6" and not c]
print("Validation:", problems or "OK")
#print("Validation:", validate_eval(eval_sample) or "OK")
print(eval_sample.test_case.value_counts().sort_index().to_dict())
eval_sample.to_csv(EVAL_DIR / "eval_sample_M1.csv", index=False)

Validation: OK
{'T1': 4, 'T2': 2, 'T3': 2, 'T4': 2, 'T5': 2, 'T6': 1}


## 6. Feasibility demo: a first local answer
A minimal TF–IDF retriever answers one question from your corpus, with response time and memory. This proves your
data loads and a local pipeline runs on your hardware. If you plan to use a local LLM or embedding model, add a cell
that loads it here and reports its size, load time and memory.

In [44]:
import tracemalloc
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tracemalloc.start()
t0 = time.perf_counter()
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english")
X = vectorizer.fit_transform(chunks.text)

def retrieve(question, k=3):
    scores = cosine_similarity(vectorizer.transform([question]), X).ravel()
    top = scores.argsort()[::-1][:k]
    return chunks.iloc[top].assign(score=scores[top])

question = eval_sample.question.iloc[0]          # TODO: try your own questions
hits = retrieve(question)
elapsed = time.perf_counter() - t0
_, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()

print("Q:", question)
print(f"Top passage [{hits.chunk_id.iloc[0]}] (score {hits.score.iloc[0]:.2f}):\n", hits.text.iloc[0][:400])
print(f"\nBuild + answer time: {elapsed*1000:.0f} ms | peak Python memory: {peak/1e6:.1f} MB | network off: {network_is_off()}")
print("\nTop 3 passages:")
for _, h in hits.iterrows():
    print(f"  [{h.chunk_id}] {h.source_location} (score {h.score:.2f}): {h.text[:120]!r}")

Q: How long is each half of a match?
Top passage [S1-p136-00] (score 0.11):
 136
The ball is in play when it is kicked and clearly moves. The kicker must not play the ball again until it has touched another player. The penalty kick is completed when the ball stops moving, goes out of play or 
the referee stops play for any offence. Additional time is allowed for a penalty kick to be taken and completed at the 
end of each half of the match or extra time. When additional ti

Build + answer time: 433 ms | peak Python memory: 3.4 MB | network off: True

Top 3 passages:
  [S1-p136-00] p. 136 (score 0.11): '136\nThe ball is in play when it is kicked and clearly moves. The kicker must not play the ball again until it has touche'
  [S1-p094-00] p. 94 (score 0.09): '94\n The fourth official indicates the minimum additional time decided by the \nreferee at the end of the final minute of '
  [S1-p093-00] p. 93 (score 0.08): '93 Law 7 | The Duration of the Match\n1. Periods of play\n A match last

**Observation.** For "How long is each half of a match?" the TF-IDF baseline retrieves the gold passage (S1, Law 7, p. 93) only at rank 3 (score 0.08): ranks 1 and 2 (pp. 136 and 94) contain the exact phrase "each half of the match" but not the answer, while the gold passage says "halves", which TF-IDF treats as a different word. On the same question, the local embedding model all-MiniLM-L6-v2 (92 MB on disk, 0.3 s load, about 9 s to encode 375 chunks, about 100 MB extra RAM, CPU only) ranks the gold passage first (score 0.66). This is one question, so it is not evidence of overall quality, but it shows the planned model runs locally on our hardware and motivates the hybrid retriever. Embeddings alone can miss exact terms such as "9.15 m" or "Law 11", so we combine them with BM25 in M2 and measure both with Recall@k and MRR on the full test set.

In [47]:
#%pip install sentence-transformers
#from sentence_transformers import SentenceTransformer
#SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2").save("models/all-MiniLM-L6-v2")
# Local embedding model check (planned for the improved system). Loaded from disk, no network.
import psutil
from sentence_transformers import SentenceTransformer

MODEL_DIR = PROJECT / "models" / "all-MiniLM-L6-v2"
proc = psutil.Process()
ram_before = proc.memory_info().rss

t0 = time.perf_counter()
embedder = SentenceTransformer(str(MODEL_DIR), device="cpu")
load_s = time.perf_counter() - t0

t0 = time.perf_counter()
emb = embedder.encode(chunks.text.tolist(), batch_size=32, show_progress_bar=False, normalize_embeddings=True)
encode_s = time.perf_counter() - t0

size_mb = sum(f.stat().st_size for f in MODEL_DIR.rglob("*") if f.is_file()) / 1e6
ram_mb = (proc.memory_info().rss - ram_before) / 1e6

q_emb = embedder.encode([question], normalize_embeddings=True)
sims = (emb @ q_emb.T).ravel()
print(f"Model: all-MiniLM-L6-v2 | size on disk: {size_mb:.0f} MB | load time: {load_s:.1f} s")
print(f"Encoded {len(chunks)} chunks in {encode_s:.1f} s | extra RAM: {ram_mb:.0f} MB | network off: {network_is_off()}")
print("\nQ:", question)
for i in sims.argsort()[::-1][:3]:
    print(f"  [{chunks.chunk_id.iloc[i]}] {chunks.source_location.iloc[i]} (score {sims[i]:.2f}): {chunks.text.iloc[i][:120]!r}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model: all-MiniLM-L6-v2 | size on disk: 92 MB | load time: 0.3 s
Encoded 375 chunks in 9.1 s | extra RAM: 106 MB | network off: False

Q: How long is each half of a match?
  [S1-p093-00] p. 93 (score 0.66): '93 Law 7 | The Duration of the Match\n1. Periods of play\n A match lasts for two equal halves of 45 minutes, which may onl'
  [S1-p093-01] p. 93 (score 0.58): 'Competition rules must state the duration of \nthe half-time interval and it may be altered only with the referee’s permi'
  [S1-p094-00] p. 94 (score 0.52): '94\n The fourth official indicates the minimum additional time decided by the \nreferee at the end of the final minute of '


## 7. Planned architecture

```mermaid
flowchart LR
  A[IFAB PDF S1, optional S2] --> B[pypdf text per page] --> C[clean + chunk, keep page] --> D[(BM25 + embedding index)]
  Q[User question + previous turn] --> F[Follow-up rewrite: add context from last turn] --> R[Retrieve top-k + rerank]
  D --> R
  R --> S{Confidence and scope check}
  S -- out of scope / low score --> X[Decline and say why]
  S -- answer depends on missing detail --> Y[Ask one clarifying question]
  S -- confident --> G[Generate answer from passages only] --> O[Answer + citation: source, Law, page]
  O --> L[(Log: query, passage IDs, answer, time)]
```

| Component | Input | Output | Chosen method | Alternative considered and why not |
|---|---|---|---|---|
| Data pipeline | IFAB PDFs | Chunks with source_id and page (M2: Law number as a separate field) | pypdf extraction, header cleanup, chunks (M2: split by Law heading) | OCR: unnecessary, the PDFs have a text layer |
| Retrieval | Question (+ previous turn for T6) | Top-k chunks with scores | Baseline TF-IDF; improved hybrid BM25 + sentence embeddings, then a cross-encoder reranker | Embeddings only: miss exact terms like "9.15 m" or "Law 11" |
| Response generation | Question + top chunks | Answer with citation | Baseline: extractive top passage; improved: small quantized local LLM prompted to answer only from the passages and cite them | Larger 7B+ model: too slow CPU-only on 16.5 GB RAM |
| Clarify / decline logic | Scores, question, previous turn | answer / clarify / decline | Score threshold tuned on the dev set + out-of-scope keyword list + per-topic "required details" (e.g. offside needs position and restart type) | LLM self-judgement only: unreliable, hard to evaluate |
| Evaluation harness | Dev/test CSV + system outputs | Metric tables, error list | Python script computing the Section 8 metrics from logged outputs | Manual only: not repeatable |

**Planned local models (Rule 1, candidates to confirm in M2):**

| Role | Model | Size | Quantization | Licence |
|---|---|---|---|---|
| Embeddings | sentence-transformers/all-MiniLM-L6-v2 | 22M params (92 MB on disk) | none (fp32) | Apache-2.0 |
| Reranker | cross-encoder/ms-marco-MiniLM-L-6-v2 | 22M params | none | Apache-2.0 |
| Generator (optional) | Llama 3.2 3B Instruct via Ollama | ~2 GB | Q4_K_M | Llama 3.2 Community License |

**Planned baseline:** TF-IDF retrieval returning the top passage with its citation (Section 6).
**Planned improved system:** hybrid retrieval + reranker + local LLM with citations + follow-up handling + tuned clarify/decline.

**Controlled experiments (one change each):** (1) chunking: fixed 150 words vs Law/sub-heading sections; (2) k = 3 vs 5 passages; (3) reranker on vs off.
**Supporting test cases:** settings in `config.yaml` so k, chunk size, threshold and model change with no code edit (T8); every query logged to JSONL with passage IDs, answer and response time (T9); command-line interface that starts offline (T10).

## 8. Evaluation plan
Baseline and improved system run on the same frozen test set (≥ 50 questions)on the same Windows machine. All metrics computed locally.

| Metric | Target on the test set | How we will compute it |
|---|---|---|
| Answer correctness (0/1/2 human score) | ≥ 75% of max score; ≥ 60% fully correct | Two members score each answerable item; report total ÷ (2 × items) and % fully correct |
| Exact match / token F1 | F1 ≥ 0.5 | SQuAD-style normalisation (case, punctuation, articles); mean EM and mean F1 |
| ROUGE-L | F1 ≥ 0.4 | `rouge-score` package |
| BERTScore | F1 ≥ 0.85 | `bert-score` with a locally downloaded model |
| Retrieval Recall@k and MRR | Recall@5 ≥ 0.85, MRR ≥ 0.6 | Gold passage labelled per question; Recall@1, @3, @5 and MRR |
| Faithfulness | ≥ 90% of claims supported | Manual check that each claim is in the cited passage (also covers T7, 20 answers) |
| Abstention and clarification | ≥ 80% declined, ≥ 70% clarified, ≤ 10% wrongly refused | % unanswerable correctly declined, % ambiguous correctly clarified, % answerable wrongly refused |
| Scorer agreement (Cohen's kappa) | κ ≥ 0.6 | Two members score ≥ 10 test outputs independently; `sklearn.metrics.cohen_kappa_score`; disagreements settled from the source |
| Response time and memory | Median < 3 s, p95 < 8 s, peak RAM < 8 GB | `time.perf_counter` per query, `psutil` peak RAM; hardware stated |

Failures will be labelled by cause (retrieval miss, chunk boundary, hallucination, ambiguous source, wrong clarify/decline) to drive at least two error-based changes.

## 9. Work plan
Three members, so one member owns two components: Abdullah owns the data pipeline and the evaluation harness (data work is concentrated in weeks 4–5, evaluation work in weeks 6–9), Yousef owns retrieval, and Omar owns response generation. Every task is reviewed by a member who does not own it.

| Task | Owner | Reviewer | Due (week) | Status |
|---|---|---|---|---|
| Collect, extract and clean corpus (S1; S2 if needed) | Abdullah | Yousef | 5 | S1 loaded: 260 pages, 375 chunks |
| Write and check 100 eval questions: T1 40, T2 20, T3 15 pairs, T4 10, T5 10, T6 5 dialogues | all members | cross-check (each member checks another member's questions) | 6 | 13 drafted and checked |
| Split dev/test, freeze test set (≥ 50) with date | Abdullah | Omar | 6 | Not started |
| Baseline TF-IDF retriever + citations | Yousef | Omar | 6 | Prototype done |
| Config file (T8), logging (T9), CLI (T10) | Omar | Yousef | 7 | Not started |
| Improved retrieval: hybrid BM25 + embeddings, reranker | Yousef | Abdullah | 8 | Embedding model tested locally |
| Improved generation: local LLM with citations, clarify/decline, follow-ups | Omar | Abdullah | 8 | Not started |
| Evaluation harness and metrics | Abdullah | Yousef | 8 | Not started |
| Controlled experiments + error analysis and two changes | all members | all members | 9 | Not started |
| M2 notebook report and README | all members | all members | 10 | Not started |

## 10. Learning plans (one paragraph per member)
Each paragraph: one gap in my knowledge — the resources I will use — what I will try — how it applies to my component.

- **Abdullah Al-Sharafi (data pipeline + evaluation harness):** My main gap is that I have only chunked text by a fixed number of words, and I do not yet know how to split a structured document by its own headings. Our M1 pipeline reads the IFAB PDF page by page and produces 375 chunks from 186 pages with text, but it has clear weaknesses: every chunk starts with its page number, the two contents pages became chunks that match many questions without answering any, and a rule that continues over a page break is cut in two. I also have a gap on the evaluation side: I know accuracy and F1 from classification, but I have never computed retrieval metrics or agreement between two human scorers. To close the first gap I will read the pypdf documentation on text extraction and study how the Laws are laid out (Law number, numbered sub-headings, bullet lists), then write regular expressions that detect "Law N" and sub-heading lines and store them as separate fields. To close the second gap I will read Chapter 8 (Evaluation in information retrieval) of Manning, Raghavan and Schütze, *Introduction to Information Retrieval*, the SQuAD paper (Rajpurkar et al., 2016) for exact match and token F1, and the scikit-learn documentation for `cohen_kappa_score`. In week 5 I will try both chunking methods on Laws 11 and 12 and compare, by hand, whether the gold passage for our 13 development questions stays in one chunk. In week 6 I will implement Recall@k and MRR and test them on small examples where I can calculate the answer on paper first. This applies directly to my components: heading-based chunks let the chatbot cite "Law 11, section 2, p. 109" instead of only a page, and a tested metric script is what makes our baseline and improved systems comparable in M2. I also need to learn how to keep paraphrase pairs and dialogue turns in the same split so the frozen test set does not leak into development.

- **Yousef Roudani (retrieval):** My gap is that I understand TF-IDF as a formula, but I do not understand well why it fails on some questions or how modern retrievers fix this. Our M1 feasibility demo showed the problem clearly: for "How long is each half of a match?" TF-IDF placed the correct passage (Law 7, p. 93) only third, because the passage says "halves" and the question says "half", and all three scores were low and close together (0.08 to 0.11). The embedding model all-MiniLM-L6-v2 placed the same passage first, but I cannot yet explain how it does that, and I do not know how to combine two retrievers or what a reranker really computes. I will read Chapter 6 (term weighting and the vector space model) and Chapter 11 (probabilistic retrieval, including BM25) of *Introduction to Information Retrieval*, the Sentence-Transformers documentation at sbert.net, especially the "Retrieve & Re-Rank" guide that explains bi-encoders and cross-encoders, and the short paper on Reciprocal Rank Fusion (Cormack, Clarke and Büttcher, 2009). In week 6 I will implement BM25 with the `rank_bm25` package and compare it with our TF-IDF baseline on the development questions, also testing whether simple stemming fixes the "half/halves" case. In week 7 I will build the embedding index, combine the two ranked lists with Reciprocal Rank Fusion, and then add the cross-encoder reranker on the top 20 passages. I will change one thing at a time and record Recall@1, Recall@3, Recall@5 and MRR after each change, so I know which step actually helps. This applies to my component because football questions mix exact terms such as "9.15 m" or "Law 11", where keyword matching is strong, with informal wording such as "pen" or "the wall", where meaning matters more. I also want to learn how retrieval scores can be used as a confidence signal, because our clarify and decline logic depends on them.

- **Omar Ahmed AL-Hamimi (response generation):** My gap is that I have used large language models only through online chat tools, and I have never run one locally or controlled what it is allowed to say. For this project the generator must run offline on a CPU-only laptop with 16.5 GB of RAM, answer only from the retrieved passages, and cite the source, Law and page. I do not yet know how quantization affects quality and speed, how to write a prompt that keeps the model grounded, or how to decide in code when the system should answer, ask a clarifying question, or decline. I will read the Ollama documentation to learn how to install and call a local model, the model card for Llama 3.2 3B Instruct to understand its prompt format and limits, and the original retrieval-augmented generation paper (Lewis et al., 2020) for the main idea of conditioning generation on retrieved text. I will also read the chapter on question answering and retrieval-augmented generation in Jurafsky and Martin, *Speech and Language Processing* (3rd edition draft). In week 7 I will first build the simple extractive baseline, which returns the best sentence from the top passage with its citation, so that I have something to compare with. In week 8 I will run the quantized model on our development questions with two or three prompt versions and check by hand whether each claim in the answer is supported by the cited passage, recording response time for each. I will then tune the decline threshold on the development set only, and write a small table of "required details" per topic, for example that an offside question needs the player's position and the type of restart. This applies to my component because a fluent answer that invents a rule is worse for our users than no answer. I also need to learn how to rewrite a follow-up such as "And in extra time?" into a full question using the previous turn.

## Attribution and AI use (Rule 2)
- Notebook structure, hardware and network cells, `validate_eval` and the TF-IDF demo come from the instructor's M1 template.
- Section 4 chunking is adapted from the template's `chunk_text`: we read the PDF page by page, keep the page number, drop running headers and skip near-empty pages.
- Generative AI (Claude) was used during development to: draft the 13 evaluation questions and reference answers (marked `ai_drafted=True`, then checked against the PDF by the member named in `checked_by`); suggest the `stop_words="english"` change and the top-3 printout in Section 6; write the local embedding check cell in Section 6; and help word Sections 1 and 7–10, including first drafts of the learning plans, which each member reviewed and edited.
- The chatbot itself uses no online model: every cell runs with the network switched off.
- Local model: sentence-transformers/all-MiniLM-L6-v2 (22M parameters, fp32, Apache-2.0).

## M1 checklist before submitting (INSERT X)
- [x] Task definition table and 10 example questions complete
- [x] Source inventory covers every document
- [x] Corpus loads and chunks with source IDs
- [x] At least 10 hand-checked evaluation questions pass validation
- [x] Feasibility demo runs with the network off
- [x] Architecture diagram, alternatives, baseline and evaluation plan filled in
- [x] Work plan names each member's component; learning plans written